# Mô hình CNN nhận diện biển báo – chạy đúng code `traffic_sign.py` của nhóm
Kiến trúc, cách đọc ảnh, cách chia dữ liệu, epoch (20), batch (32), bộ tối ưu (Adam) **giữ nguyên như file `traffic_sign.py`**.
Chỉ sửa các lỗi làm code không chạy được: đường dẫn Windows `'\\'`, phần kiểm tra cuối file (`resize`, `argmax`).

**Cách chạy:** Runtime → Change runtime type → **T4 GPU** → Save → **Runtime → Run all** (khoảng 10–20 phút).

Đầu ra: `traffic_classifier.h5` (dùng cho `gui.py`), `model.tflite` (chép vào app), `hinh5_duong_cong_huan_luyen.png`, `bang9_precision_recall_f1.csv`, và bảng số liệu in ở cuối ô 7.

In [ ]:
# ===== Ô 1: Cài thư viện, kiểm tra GPU =====
!pip install -q kagglehub
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
# Nếu dữ liệu nằm trên Google Drive: bỏ dấu # ở 2 dòng dưới
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ===== Ô 2: Tải dữ liệu GTSRB =====
import os, glob, time
import numpy as np, pandas as pd
from PIL import Image

DATA_DIR = None   # Nếu đã có GTSRB trên Drive: DATA_DIR = '/content/drive/MyDrive/GTSRB'
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download('meowmeowmeowmeowmeow/gtsrb-german-traffic-sign')
ROOT = None
for d, subdirs, files in os.walk(DATA_DIR):
    if 'Train' in subdirs and 'Test.csv' in files:
        ROOT = d; break
assert ROOT, 'Không tìm thấy thư mục chứa Train/ và Test.csv trong ' + DATA_DIR
print('Thư mục dữ liệu:', ROOT)

In [ ]:
# ===== Ô 3: Truy xuất hình ảnh và nhãn (giữ nguyên cách làm của traffic_sign.py) =====
data, labels = [], []
classes = 43
for i in range(classes):
    path = os.path.join(ROOT, 'Train', str(i))
    for a in os.listdir(path):
        try:
            image = Image.open(os.path.join(path, a)).convert('RGB')   # sửa: dùng os.path.join thay cho '\\'
            image = image.resize((30, 30))
            data.append(np.array(image))
            labels.append(i)
        except Exception:
            print('Load ảnh lỗi:', a)
data = np.array(data)
labels = np.array(labels)
print(data.shape, labels.shape)

In [ ]:
# ===== Ô 4: Tách dữ liệu và xây dựng mô hình CNN (nguyên văn traffic_sign.py) =====
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout

X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
y_train = to_categorical(y_train, 43)
y_test = to_categorical(y_test, 43)

model = Sequential()
model.add(Conv2D(filters=32, padding='same', kernel_size=(5, 5), activation='relu', input_shape=X_train.shape[1:]))
model.add(Conv2D(filters=32, padding='same', kernel_size=(5, 5), activation='relu'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(rate=0.25))
model.add(Conv2D(filters=64, padding='same', kernel_size=(3, 3), activation='relu'))
model.add(Conv2D(filters=64, padding='same', kernel_size=(3, 3), activation='relu'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(rate=0.25))
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(rate=0.5))
model.add(Dense(43, activation='softmax'))
model.summary()   # Tổng tham số: 897,611

epochs = 20
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
print('Start training')
t0 = time.time()
history = model.fit(X_train, y_train, batch_size=32, epochs=epochs, validation_data=(X_test, y_test))
train_min = (time.time() - t0) / 60
model.save('traffic_classifier.h5')      # lưu file train (dùng cho gui.py)
model.save('traffic_classifier.keras')

In [ ]:
# ===== Ô 5: Đồ thị độ chính xác và mất mát (Hình 5) =====
import matplotlib.pyplot as plt
ep = range(1, len(history.history['loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(ep, history.history['accuracy'], 'o-', label='Huấn luyện'); ax[0].plot(ep, history.history['val_accuracy'], 's-', label='Kiểm định')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Độ chính xác'); ax[0].set_title('(a) Độ chính xác'); ax[0].grid(alpha=.3); ax[0].legend()
ax[1].plot(ep, history.history['loss'], 'o-', label='Huấn luyện'); ax[1].plot(ep, history.history['val_loss'], 's-', label='Kiểm định')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Hàm mất mát'); ax[1].set_title('(b) Hàm mất mát'); ax[1].grid(alpha=.3); ax[1].legend()
plt.tight_layout(); plt.savefig('hinh5_duong_cong_huan_luyen.png', dpi=200); plt.show()

In [ ]:
# ===== Ô 6: Kiểm tra trên tập Test.csv + Precision/Recall/F1 từng lớp (Bảng 9) =====
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
test = pd.read_csv(os.path.join(ROOT, 'Test.csv'))
X_all = np.array([np.array(Image.open(os.path.join(ROOT, p)).convert('RGB').resize((30, 30))) for p in test['Path']])
y_all = test['ClassId'].values
pred_all = model.predict(X_all, verbose=0).argmax(axis=1)          # sửa: lấy argmax trước khi so sánh
acc_all = accuracy_score(y_all, pred_all)

idx = test.sample(min(8000, len(test)), random_state=42).index.values   # 8.000 ảnh ngẫu nhiên như báo cáo
y8, p8 = y_all[idx], pred_all[idx]
acc8 = accuracy_score(y8, p8)
P, R, F, S = precision_recall_fscore_support(y8, p8, labels=list(range(43)), zero_division=0)
bang9 = pd.DataFrame({'Nhãn': range(43), 'Số ảnh': S, 'Precision': P.round(2), 'Recall': R.round(2), 'F1': F.round(2)})
bang9.to_csv('bang9_precision_recall_f1.csv', index=False)
print(bang9.to_string(index=False))

In [ ]:
# ===== Ô 7: Chuyển sang TensorFlow Lite cho app điện thoại =====
conv = tf.lite.TFLiteConverter.from_keras_model(model)
open('model_float32.tflite', 'wb').write(conv.convert())
conv = tf.lite.TFLiteConverter.from_keras_model(model); conv.optimizations = [tf.lite.Optimize.DEFAULT]
open('model.tflite', 'wb').write(conv.convert())

it = tf.lite.Interpreter(model_path='model.tflite'); it.allocate_tensors()
inp, out = it.get_input_details()[0]['index'], it.get_output_details()[0]['index']
pq, ts = [], []
for x in X_all[idx].astype(np.float32):
    t = time.perf_counter(); it.set_tensor(inp, x[None]); it.invoke(); ts.append((time.perf_counter() - t) * 1000)
    pq.append(int(it.get_tensor(out).argmax()))
acc_q = accuracy_score(y8, pq)
mb = lambda f: os.path.getsize(f) / 1024 / 1024

print('\n================ SỐ LIỆU ĐIỀN VÀO BÁO CÁO ================')
print(f'Bảng 8  | Epoch / batch: {epochs} / 32 | Adam (tốc độ học mặc định 0.001) | Thời gian huấn luyện: {train_min:.1f} phút')
print(f'Bảng 8  | Tập huấn luyện / kiểm định: {len(X_train)} / {len(X_test)} ảnh | GPU: {tf.config.list_physical_devices("GPU")}')
print(f'Hình 5  | Epoch cuối: acc huấn luyện {history.history["accuracy"][-1]*100:.2f}% | acc kiểm định {history.history["val_accuracy"][-1]*100:.2f}%')
print(f'Mục 3   | Toàn bộ {len(y_all)} ảnh Test: {acc_all*100:.2f}% | 8.000 ảnh ngẫu nhiên: {acc8*100:.2f}% ({int((y8==p8).sum())}/{len(y8)} ảnh đúng)')
print(f'Mục 3   | Macro P / R / F1 (8.000 ảnh): {P.mean():.2f} / {R.mean():.2f} / {F.mean():.2f}')
print(f'Mục 5.1 | .h5 {mb("traffic_classifier.h5"):.2f} MB | TFLite float32 {mb("model_float32.tflite"):.2f} MB | TFLite lượng tử hóa {mb("model.tflite"):.2f} MB')
print(f'Mục 5.1 | Độ chính xác TFLite lượng tử hóa (8.000 ảnh): {acc_q*100:.2f}% | thời gian trên CPU Colab: {np.mean(ts):.2f} ms/ảnh')

In [ ]:
# ===== Ô 8 (tùy chọn): Thử trên ảnh biển báo thực tế tại Việt Nam =====
# Nén thư mục ảnh thành anh_thuc_te.zip, bên trong mỗi thư mục con mang SỐ NHÃN (0–42), ví dụ anh_thuc_te/14/stop1.jpg
# rồi kéo thả file zip vào khung Files bên trái Colab và chạy ô này.
import zipfile
if os.path.exists('anh_thuc_te.zip'):
    zipfile.ZipFile('anh_thuc_te.zip').extractall('.')
    rows = []
    for f in glob.glob('anh_thuc_te/*/*'):
        lab = int(os.path.basename(os.path.dirname(f)))
        x = np.array(Image.open(f).convert('RGB').resize((30, 30)))[None]
        pr = model.predict(x, verbose=0)[0]
        rows.append({'Ảnh': os.path.basename(f), 'Nhãn đúng': lab, 'Dự đoán': int(pr.argmax()), 'Độ tin cậy': round(float(pr.max()), 3)})
    thuc_te = pd.DataFrame(rows)
    dung = (thuc_te['Nhãn đúng'] == thuc_te['Dự đoán']).sum()
    print(f'Ảnh thực tế: đúng {dung}/{len(thuc_te)} ({dung/len(thuc_te)*100:.1f}%)')
    print(thuc_te[thuc_te['Nhãn đúng'] != thuc_te['Dự đoán']].to_string(index=False))
    thuc_te.to_csv('ket_qua_anh_thuc_te.csv', index=False)
else:
    print('Chưa có anh_thuc_te.zip – bỏ qua ô này.')

In [ ]:
# ===== Ô 9: Tải kết quả về máy =====
try:
    from google.colab import files
    for f in ['traffic_classifier.h5', 'model.tflite', 'hinh5_duong_cong_huan_luyen.png', 'bang9_precision_recall_f1.csv']:
        files.download(f)
    if os.path.exists('ket_qua_anh_thuc_te.csv'):
        files.download('ket_qua_anh_thuc_te.csv')
except ImportError:
    pass